In [ ]:
!pip install -q -U gradio
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-huggingface
!pip install -q transformers accelerate
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import gradio as gr
from llama_index.readers.file import PDFReader
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime

In [ ]:
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = HuggingFaceLLM(
    model_name="microsoft/phi-2",
    tokenizer_name="microsoft/phi-2",
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    device_map="auto",
    model_kwargs={"torch_dtype": torch.float16, "trust_remote_code": True},
    tokenizer_kwargs={"trust_remote_code": True},
)

pdf_loader = PDFReader()
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
last_index = None
last_llama_documents = None

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (16 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [ ]:
def build_index_from_pdfs(files):
    """Load one or more PDFs, chunk each, and build a single queryable
     vector index spanning all of them. Each chunk keeps source_file AND
    page_number metadata, so answers can be traced back to the exact file
    + page they came from.

    Returns (query_engine, status_message). query_engine is None if
    something went wrong, so the chat function knows not to proceed.
    """
    global last_index, last_llama_documents

    if not files:
        return None, "⚠️ Please upload at least one PDF."

    try:
        all_llama_documents = []
        per_file_summary = []

        for file in files:
            filename = file.name.split("/")[-1]
            # file.name is the temp filepath Gradio saves each upload to
            documents = pdf_loader.load_data(file.name)
            file_chunk_count = 0
            # Chunk each page SEPARATELY (instead of joining the whole PDF
            # into one string first) so every chunk can be tagged with the
            # real page number it came from, for citation purposes below.
            for page_num, page_doc in enumerate(documents, start=1):
                page_chunks = splitter.split_text(page_doc.text)
                for i, chunk in enumerate(page_chunks):
                    all_llama_documents.append(
                        Document(
                            text=chunk,
                            metadata={
                                "chunk_index": i,
                                "source_file": filename,
                                "page_number": page_num,
                            },
                            # Metadata stays attached to the node for OUR
                            # code (citations, eval metrics) but is hidden
                            # from both the embedding model and the LLM's
                            # prompt - otherwise Phi-2 sometimes echoes the
                            # metadata block back instead of answering.
                            excluded_llm_metadata_keys=["chunk_index", "source_file", "page_number"],
                            excluded_embed_metadata_keys=["chunk_index", "source_file", "page_number"],
                        )
                    )
        file_chunk_count += len(page_chunks)
        per_file_summary.append(f"**{filename}** ({len(documents)} page(s), {file_chunk_count} chunk(s))")

        index = VectorStoreIndex.from_documents(all_llama_documents)
        query_engine = index.as_query_engine(similarity_top_k=4)

        last_index = index
        last_llama_documents = all_llama_documents

        status = (
            f"✅ Processed {len(files)} file(s):<br>" + "<br>".join(per_file_summary) +
            f"<br>{len(all_llama_documents)} total chunk(s) indexed. Ask a question below!"
        )
        return query_engine, status

    except Exception as e:
        return None, f"❌ Failed to process PDF: {e}"

In [ ]:
def clean_phi2_response(text):
    """Phi-2 is a base (non-chat-tuned) model, so after it answers it
    often keeps generating - continuing into a fake StackOverflow-style
    'A:' turn, a new 'Q:', or a code snippet, because that pattern shows
    up a lot in its training data. Trim the response down to just the
    first coherent answer so that leftover text never reaches the chat.
    """
    text = text.strip()

    # Cut off as soon as the model starts hallucinating a new turn/code block
    stop_markers = ["\nA:", "\nQ:", "\n```", "```", "\nimport ", "\nwith open(", "\ndef "]
    cut_index = len(text)
    for marker in stop_markers:
        idx = text.find(marker)
        if idx != -1:
            cut_index = min(cut_index, idx)
    text = text[:cut_index].strip()

    # If it's still a rambling multi-paragraph block, keep just the first paragraph
    text = text.split("\n\n")[0].strip()

    return text

In [ ]:
def respond(message, chat_history, query_engine):
    """Answer a user question using the current index's query_engine,
    then append the exchange to the chat history (messages format)."""
    if not message.strip():
        return chat_history, ""

    if query_engine is None:
        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "Please upload and process at least one PDF before asking questions."}
        ]
        return chat_history, ""

    response = query_engine.query(message)

    citations = sorted({
        (node.metadata.get("source_file", "unknown"), node.metadata.get("page_number", "?"))
        for node in getattr(response, "source_nodes", [])
    })
    answer = clean_phi2_response(str(response))
    if citations:
        ref_lines = "\n".join(f"- {fname}, page {page}" for fname, page in citations)
        answer += f"\n\n**References:**\n{ref_lines}"

    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return chat_history, ""

In [ ]:
def export_chat_history(chat_history):
    """Write the current conversation to a .txt file so gr.DownloadButton
    can hand it to the browser. Runs each time the download button is
    clicked, so it always reflects the latest chat."""
    if not chat_history:
        gr.Warning("There's no chat history yet — ask a question first.")
        return gr.update()  # leave the download button's file unchanged

    lines = []
    for turn in chat_history:
        speaker = "You" if turn["role"] == "user" else "Bot"
        lines.append(f"{speaker}: {turn['content']}")
    transcript = "\n\n".join(lines)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = f"/content/chat_history_{timestamp}.txt"
    with open(filepath, "w") as f:
        f.write(transcript)

    return filepath

In [ ]:
with gr.Blocks(title="QuickSearch with Pfizer", theme=gr.themes.Default(primary_hue="blue")) as demo:
    # query_engine lives in gr.State so it persists between interactions
    # within a session without becoming a global variable.
    query_engine_state = gr.State(None)

    gr.Markdown("# 📄🧬🦠 💊 QuickSearch with Pfizer")
    gr.Markdown("Upload one or more PDFs and ask a question. The chatbot will perform a QuickSearch to find the answer.")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_upload = gr.File(
                label="Upload PDF(s)",
                file_types=[".pdf"],
                file_count="multiple",
            )
            process_btn = gr.Button("Process Document(s)", variant="primary")
            status_box = gr.Markdown("No documents loaded yet.")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=450)
            question_box = gr.Textbox(
                label="Ask a question",
                placeholder="e.g. What are the storage conditions?",
                lines=1,
            )
        with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                download_btn = gr.DownloadButton("⬇️ Download Chat History")

    # --- Wiring ---
    process_btn.click(
        fn=build_index_from_pdfs,
        inputs=[pdf_upload],
        outputs=[query_engine_state, status_box],
    )

    send_btn.click(
        fn=respond,
        inputs=[question_box, chatbot, query_engine_state],
        outputs=[chatbot, question_box],
    )
    question_box.submit(
        fn=respond,
        inputs=[question_box, chatbot, query_engine_state],
        outputs=[chatbot, question_box],
    )

    download_btn.click(
        fn=export_chat_history,
        inputs=[chatbot],
        outputs=[download_btn],
    )

demo.launch(debug=True)

/tmp/ipykernel_1070/4197102662.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="QuickSearch with Pfizer", theme=gr.themes.Default(primary_hue="blue")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://12632a75575877c6b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CodeGenTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://12632a75575877c6b7.gradio.live
